# Visualización de una simulación TVB de una región

Este cuaderno carga el contenedor binario NumPy generado por los scripts de `src/zerlaut/` y representa una o varias runs sin mezclar sus datos.

In [ ]:
%pwd

In [ ]:
result_dir = 'results'
result_file = 'simple_simulation.npz'
# result_file = 'simulation.npz'

RUNS = [0]  # Por ejemplo, [0, 1, 2] para comparar varias runs.

# Se busca el fichero desde la raíz o desde un subdirectorio de notebooks.
from pathlib import Path
base_path = Path(result_file)
search_roots = [Path('../' * i) / result_dir for i in range(3)]

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

results = {}
metadata = {}
for run in RUNS:
    run_file = f'{base_path.stem}_run_{run}{base_path.suffix}'
    result_path = next(root / run_file for root in search_roots if (root / run_file).exists())
    with np.load(result_path, allow_pickle=False) as stored:
        results[run] = {
            name: stored[name].copy() for name in stored.files
            if name != 'metadata_json'
        }
        metadata[run] = json.loads(str(stored['metadata_json']))

print(f'Runs cargadas: {RUNS}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, constrained_layout=True)

colors = plt.get_cmap('tab10').colors
for index, run in enumerate(RUNS):
    result = results[run]
    time_s = result['time_ms'] / 1000.0
    color = colors[index % len(colors)]
    axes[0].plot(time_s, result['stimulus_hz'], color=color, label=f'Run {run}: estímulo')
    axes[0].plot(time_s, result['Fe_ext_hz'], color=color, linestyle='--', label=f'Run {run}: Fe')
    axes[0].plot(time_s, result['Fi_ext_hz'], color=color, linestyle=':', label=f'Run {run}: Fi')
    axes[1].plot(time_s, result['E_hz'], color=color, label=f'Run {run}: E')
    axes[1].plot(time_s, result['I_hz'], color=color, linestyle='--', label=f'Run {run}: I')
    axes[2].plot(time_s, result['W_e_pA'], color=color, label=f'Run {run}: W_e')
    axes[2].plot(time_s, result['W_i_pA'], color=color, linestyle='--', label=f'Run {run}: W_i')

axes[0].set_ylabel('Entrada (Hz)')
axes[0].legend(loc='upper right')

axes[1].set_ylabel('Tasa (Hz)')
axes[1].legend(loc='upper right')

axes[2].set_ylabel('Adaptación (pA)')
axes[2].set_xlabel('Tiempo (s)')
axes[2].legend(loc='upper right')

for axis in axes:
    axis.grid(alpha=0.25)

fig.suptitle(f'Zerlaut-AdEx: runs {RUNS}')
plt.show()

In [ ]:
summary = []
for run in RUNS:
    result = results[run]
    summary.append({
        'run': run,
        'duración (s)': float(result['time_ms'][-1] / 1000.0),
        'E media (Hz)': float(np.mean(result['E_hz'])),
        'E máxima (Hz)': float(np.max(result['E_hz'])),
        'I media (Hz)': float(np.mean(result['I_hz'])),
        'I máxima (Hz)': float(np.max(result['I_hz'])),
    })
summary